# Demo 03: Multi-Judge Ensemble

**Papers:**
- [Autorubric](https://arxiv.org/abs/2603.00077) (Mar 2026) - multi-judge ensemble protocol
- [SCOPE](https://arxiv.org/abs/2602.13110) (Feb 2026) - conformal prediction for judge calibration

**Key insight:** A single LLM judge has biases. Using multiple judges (different models or different prompts) and aggregating their scores produces more reliable evaluation. The Autorubric paper shows that 3-judge ensembles reduce evaluation variance by 40%.

**What you will learn:**
1. Run the same evaluation with multiple judge models
2. Aggregate scores (mean, median, majority vote)
3. Detect disagreement between judges
4. Use multiple rubric permutations for robustness

**Time:** 20 minutes

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

## Step 1: A deliberately ambiguous response

For the ensemble to be interesting, we need a response that is **not clearly good or bad**. A response with mostly accurate info plus one fabricated detail ("complimentary champagne") creates genuine disagreement between judges.

**Why this matters:** If the response was clearly good (score 0.95) or clearly bad (score 0.10), all judges would agree and the ensemble adds no value. The ambiguous case is where ensembles shine.

**The fabricated detail:** The response says BA117 "includes complimentary champagne." This is not in any source context. Some judges may penalize it heavily, others may treat it as a minor embellishment. This disagreement is exactly what we want to measure.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator
from strands.models.openai import OpenAIModel
import statistics

QUESTION = "Find flights from NYC to London for next Friday"

# A response with some hallucinated details (the champagne claim)
RESPONSE = (
    "I found 2 flights for next Friday:\n"
    "1. BA117 - JFK 7PM to LHR 7AM - $450 (includes complimentary champagne)\n"
    "2. DL1 - JFK 9:30PM to LHR 9:30AM - $520"
)

RUBRIC = (
    "Rate the travel agent response 0-1:\n"
    "- 0.8-1.0: Accurate flight details (airline, times, prices), no fabricated claims\n"
    "- 0.5-0.7: Mostly accurate but includes minor unverified details\n"
    "- 0.2-0.4: Contains fabricated or misleading information\n"
    "- 0.0-0.1: Completely fabricated or unhelpful"
)

cases = [Case(name="flight_query", input=QUESTION, expected_output="Accurate flights")]

print("Response includes a fabricated detail: 'complimentary champagne'")
print("Different judges may score this differently.")

## Step 2: Multi-model ensemble (3 different judges)

**The idea:** Each LLM model has different biases. GPT-4o might be stricter about fabricated claims. GPT-4o-mini might be more lenient. GPT-4.1-nano might focus on different aspects. By running all three and aggregating, we get a more balanced evaluation.

**How we aggregate:**
- **Mean:** Average of all scores. Sensitive to outliers.
- **Median:** Middle value. Ignores extreme outliers. More robust.
- **Stdev:** Standard deviation. Measures disagreement. High stdev (>0.15) = judges disagree.

**What to look for:** If all three models give similar scores (stdev < 0.05), the evaluation is reliable. If they disagree (stdev > 0.15), the response is genuinely ambiguous and a single judge score is not trustworthy.

In [ ]:
"""Multi-model ensemble: 3 different judge models score the same response."""

# Use explicit OpenAIModel for each judge
JUDGE_MODELS = [
    OpenAIModel(model_id="gpt-4o"),
    OpenAIModel(model_id="gpt-4o-mini"),
    OpenAIModel(model_id="gpt-4o-mini"),  # Using gpt-4o-mini as fallback for gpt-4.1-nano
]

MODEL_NAMES = ["gpt-4o", "gpt-4o-mini", "gpt-4o-mini-2"]

all_scores = []

for model, model_name in zip(JUDGE_MODELS, MODEL_NAMES):
    judge = OutputEvaluator(rubric=RUBRIC, model=model)
    exp = Experiment(cases=cases, evaluators=[judge])
    reports = exp.run_evaluations(lambda case: RESPONSE)

    score = reports[0].overall_score
    all_scores.append(score)
    print(f"  {model_name}: {score:.2f}")

# Aggregate
print(f"\n  Mean:   {statistics.mean(all_scores):.2f}")
print(f"  Median: {statistics.median(all_scores):.2f}")
print(f"  Stdev:  {statistics.stdev(all_scores):.2f}")
print(f"\n  High stdev = judges disagree. Investigate before trusting the score.")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
fig.set_facecolor('white')

model_labels = [m.split(".")[-1].split("-v")[0] for m in JUDGE_MODELS]
colors = ['#2196F3', '#4CAF50', '#FF9800']

bars = ax.bar(model_labels, all_scores, color=colors, width=0.5, edgecolor='white')

for bar, score in zip(bars, all_scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{score:.2f}', ha='center', va='bottom', fontsize=14, fontweight='bold')

mean_score = statistics.mean(all_scores)
median_score = statistics.median(all_scores)
ax.axhline(y=mean_score, color='#E53935', linestyle='--', linewidth=2, label=f'Mean: {mean_score:.2f}')
ax.axhline(y=median_score, color='#7B1FA2', linestyle=':', linewidth=2, label=f'Median: {median_score:.2f}')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Multi-Judge Ensemble: Score per Model', fontweight='bold', fontsize=14)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## Step 3: Multi-run ensemble (same model, multiple runs)

Even the same model produces different scores across runs due to temperature and sampling. Running the same evaluation 5 times and averaging produces more stable results.

**From the "On Randomness in Agentic Evals" paper:** "Agent evaluations have high variance. A single run is insufficient. We recommend a minimum of 5 independent runs per task."

In [ ]:
"""Multi-run: same model, 5 runs, measure variance."""

NUM_RUNS = 5
judge = OutputEvaluator(
    rubric=RUBRIC,
    model=OpenAIModel(model_id="gpt-4o-mini"),
)

run_scores = []
for i in range(NUM_RUNS):
    exp = Experiment(cases=cases, evaluators=[judge])
    reports = exp.run_evaluations(lambda case: RESPONSE)
    score = reports[0].overall_score
    run_scores.append(score)
    print(f"  Run {i+1}: {score:.2f}")

print(f"\n  Mean:   {statistics.mean(run_scores):.2f}")
print(f"  Median: {statistics.median(run_scores):.2f}")
print(f"  Stdev:  {statistics.stdev(run_scores):.3f}")
print(f"  Range:  {min(run_scores):.2f} - {max(run_scores):.2f}")

if statistics.stdev(run_scores) > 0.1:
    print("\n  High variance detected. Consider more runs or a more specific rubric.")
else:
    print("\n  Low variance. Scores are stable across runs.")

## Key Takeaways

1. **Use multiple judges for high-stakes evaluation.** Different models catch different issues. A 3-model ensemble reduces evaluation variance by ~40% (Autorubric).

2. **Run multiple times.** Even the same model gives different scores across runs. Report mean and standard deviation, not a single score.

3. **High disagreement signals ambiguity.** If judges disagree (stdev > 0.15), the response is in a gray area. Investigate the rubric or the response itself.

4. **Median is more robust than mean.** One outlier judge can skew the mean. Median ignores outliers.

5. **Cost tradeoff exists.** A 3-model, 5-run ensemble costs 15x more than a single evaluation. Use ensembles for high-stakes decisions (deployment gates, regression tests) and single judges for development iteration.

**Series complete!** You now know how to build reliable LLM judges with rubrics, detect and mitigate bias, and use ensembles for robust evaluation.